# Daily Challenge – Week 4, Day 2: Global Power Plant Database
**Dataset:** World Resources Institute – Global Power Plant Database v1.3 (~35,000 plants in 167 countries).
**Libraries:** NumPy (types, statistics, filtering, matrix operations), Pandas (cleaning, grouping), SciPy (hypothesis tests), Matplotlib & Seaborn (charts).

## 1. Data Import and Cleaning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("global_power_plant_database.csv", low_memory=False)
print(df.shape)
df.head(3)

In [ ]:
missing_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
missing_pct[missing_pct > 0]

**Cleaning decisions:**
- `other_fuel2`, `other_fuel3`, `wepp_id` and the yearly `generation_gwh_*` columns are mostly empty (55–99% missing) → dropped, except **`estimated_generation_gwh_2017`** (only 5% missing), which we keep as our generation measure.
- `owner` (40% missing) → filled with `"Unknown"`.
- `commissioning_year` (50% missing) → kept as missing (we can't guess when a plant was built); it's only used for the time analysis, on the plants where it's known. Values like `2004.5` (averages of several units) are rounded down to whole years with NumPy.
- Numeric columns are made sure to be numeric with NumPy/pandas conversion.

In [ ]:
keep = ["country", "country_long", "name", "capacity_mw", "latitude", "longitude",
        "primary_fuel", "other_fuel1", "commissioning_year", "owner", "estimated_generation_gwh_2017"]
df = df[keep].copy()

df["owner"] = df["owner"].fillna("Unknown")

for col in ["capacity_mw", "latitude", "longitude", "commissioning_year", "estimated_generation_gwh_2017"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float64)

df["commissioning_year"] = np.floor(df["commissioning_year"])

print(df.dtypes)
print("\nMissing after cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0])

## 2. Exploratory Data Analysis

In [ ]:
df[["capacity_mw", "commissioning_year", "estimated_generation_gwh_2017"]].agg(["mean", "median", "std"]).round(1)

In [ ]:
print("Plants by fuel type:")
print(df["primary_fuel"].value_counts())

print("\nTop 10 countries by number of plants:")
print(df["country_long"].value_counts().head(10))

In [ ]:
fuel_summary = (df.groupby("primary_fuel")["capacity_mw"]
                  .agg(plants="count", total_mw="sum", mean_mw="mean", median_mw="median")
                  .sort_values("total_mw", ascending=False))
fuel_summary["share_of_capacity_%"] = (fuel_summary["total_mw"] / fuel_summary["total_mw"].sum() * 100).round(1)
fuel_summary.round(1)

## 3. Statistical Analysis of Power Output by Fuel Type

In [ ]:
main_fuels = ["Coal", "Gas", "Hydro", "Nuclear", "Wind", "Solar", "Oil"]

rows = []
for fuel in main_fuels:
    cap = df.loc[df["primary_fuel"] == fuel, "capacity_mw"].to_numpy()
    rows.append({"fuel": fuel,
                 "mean": np.mean(cap), "median": np.median(cap), "std": np.std(cap),
                 "p90": np.percentile(cap, 90)})
pd.DataFrame(rows).set_index("fuel").round(1)

### Hypothesis tests
- **Test 1 (all main fuels):** H0 = the mean plant capacity is the same for all fuel types. Capacities are very skewed, so besides the classic **one-way ANOVA** I also run the **Kruskal-Wallis** test, which doesn't assume normal data.
- **Test 2 (two fuels):** H0 = coal and gas plants have the same mean capacity → **Welch t-test**.

In [ ]:
groups = [df.loc[df["primary_fuel"] == f, "capacity_mw"] for f in main_fuels]

f_stat, p_anova = stats.f_oneway(*groups)
h_stat, p_kw = stats.kruskal(*groups)
print(f"ANOVA:          F = {f_stat:.1f}, p = {p_anova:.2e}")
print(f"Kruskal-Wallis: H = {h_stat:.1f}, p = {p_kw:.2e}")

coal = df.loc[df["primary_fuel"] == "Coal", "capacity_mw"]
gas = df.loc[df["primary_fuel"] == "Gas", "capacity_mw"]
t_stat, p_t = stats.ttest_ind(coal, gas, equal_var=False)
print(f"Coal vs Gas (Welch t-test): t = {t_stat:.1f}, p = {p_t:.2e}")

## 4. Time Series Analysis – how the fuel mix evolved
(Only the ~50% of plants with a known commissioning year.)

In [ ]:
dated = df.dropna(subset=["commissioning_year"]).copy()
dated["decade"] = (dated["commissioning_year"].to_numpy() // 10 * 10).astype(int)
dated = dated[dated["decade"] >= 1950]

capacity_by_decade = dated.pivot_table(index="decade", columns="primary_fuel",
                                       values="capacity_mw", aggfunc="sum", fill_value=0)
share_by_decade = capacity_by_decade.div(capacity_by_decade.sum(axis=1), axis=0) * 100
share_by_decade[main_fuels].round(1)

In [ ]:
share_by_decade[main_fuels].plot(kind="bar", stacked=True, figsize=(11, 5), colormap="tab10")
plt.title("Share of NEW capacity built each decade, by fuel type (%)")
plt.ylabel("% of capacity commissioned")
plt.xlabel("Decade")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Advanced Visualization

In [ ]:
plt.figure(figsize=(14, 7))
sns.scatterplot(data=df[df["primary_fuel"].isin(main_fuels)], x="longitude", y="latitude",
                hue="primary_fuel", size="capacity_mw", sizes=(2, 200), alpha=0.5, linewidth=0)
plt.title("Geographical distribution of power plants (size = capacity)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fuel_summary["total_mw"].head(8).plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Total capacity by fuel (MW)")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df[df["primary_fuel"].isin(main_fuels)], x="primary_fuel", y="capacity_mw",
            order=main_fuels, ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Plant capacity by fuel (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
top_countries = df.groupby("country_long")["capacity_mw"].sum().sort_values(ascending=False).head(10)
top_countries.sort_values().plot(kind="barh", figsize=(9, 5), color="darkorange")
plt.title("Top 10 countries by total installed capacity (MW)")
plt.xlabel("MW")
plt.show()

## 6. Matrix Operations in a Real-World Context
We build the **correlation matrix** of numeric attributes (capacity, generation, location, year, plus "is it a fossil-fuel plant") and compute its **eigenvalues and eigenvectors** with NumPy.

In [ ]:
mat_df = df.dropna(subset=["commissioning_year", "estimated_generation_gwh_2017"]).copy()
mat_df["is_fossil"] = np.isin(mat_df["primary_fuel"], ["Coal", "Gas", "Oil", "Petcoke"]).astype(int)
mat_df["log_capacity"] = np.log10(mat_df["capacity_mw"])
mat_df["log_generation"] = np.log10(mat_df["estimated_generation_gwh_2017"].clip(lower=0.01))

cols = ["log_capacity", "log_generation", "latitude", "longitude", "commissioning_year", "is_fossil"]
X = mat_df[cols].to_numpy()

# Standardize with NumPy, then correlation matrix = (Xᵀ X) / (n − 1)
X_std = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
corr = (X_std.T @ X_std) / (len(X_std) - 1)
pd.DataFrame(corr, index=cols, columns=cols).round(2)

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(corr)
order = np.argsort(eigenvalues)[::-1]          # largest first
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

print("Eigenvalues:", eigenvalues.round(2))
print("Share of total variance:", (eigenvalues / eigenvalues.sum() * 100).round(1), "%")
pd.DataFrame(eigenvectors[:, :2], index=cols, columns=["Eigenvector 1", "Eigenvector 2"]).round(2)

**Why eigenvalues and eigenvectors matter here:**
- Each **eigenvector** is a "direction" that combines the attributes; its **eigenvalue** says how much of the total variation in the data lies along that direction. This is exactly what **PCA** does.
- The first eigenvector loads mainly on **capacity and generation** together (they move together: big plants generate more) – it's essentially a "plant size" factor.
- If one eigenvalue is much bigger than the rest, a few combined factors explain most of the data, so we could describe each plant with fewer numbers (dimensionality reduction).
- Eigenvalues close to 0 would mean some attributes are almost exact combinations of others (redundant information).

## 7. Integrating NumPy with Pandas and Matplotlib

In [ ]:
# a) NumPy for complex filtering in Pandas: large renewable plants commissioned after 2010
mask = (np.isin(df["primary_fuel"], ["Solar", "Wind", "Hydro"])
        & (df["capacity_mw"].to_numpy() > 500)
        & (df["commissioning_year"].to_numpy() > 2010))
print("Large new renewable plants:", mask.sum())
print(df.loc[mask, "country_long"].value_counts().head(5))

# b) np.select to create size categories
conditions = [df["capacity_mw"] < 10, df["capacity_mw"] < 100, df["capacity_mw"] < 1000]
df["size_class"] = np.select(conditions, ["Small (<10 MW)", "Medium (10–100)", "Large (100–1000)"],
                             default="Very large (≥1000)")
df["size_class"].value_counts()

In [ ]:
# c) NumPy for a more advanced plot: log-spaced bins for a very skewed distribution
bins = np.logspace(0, np.log10(df["capacity_mw"].max()), 50)
plt.figure(figsize=(10, 4))
for fuel in ["Solar", "Wind", "Gas", "Coal"]:
    plt.hist(df.loc[df["primary_fuel"] == fuel, "capacity_mw"], bins=bins, alpha=0.5, label=fuel)
plt.xscale("log")
plt.title("Capacity distribution with NumPy log-spaced bins")
plt.xlabel("Capacity (MW, log scale)")
plt.ylabel("Number of plants")
plt.legend()
plt.show()

## 8. Report – Key Insights
1. **Scale:** ~35,000 plants in 167 countries, with a total capacity of about **5.7 million MW**. China (~1.4M MW) and the USA (~1.2M MW) together hold almost half of it; the USA has by far the most *plants* (~9,800), but many are small.
2. **Many plants ≠ much power:** Solar is the most common plant type (10,665 plants) but only **3.3%** of capacity, because most solar plants are tiny (median 5.8 MW). **Coal (34%) and gas (26%)** still dominate installed capacity, followed by hydro (19%).
3. **Very different plant sizes:** nuclear plants are the biggest on average (median ~1,900 MW), then coal (600 MW). The largest single plant is China's Three Gorges Dam (22,500 MW, hydro). Capacity is extremely skewed, so medians and log scales are needed to compare fuels.
4. **Statistics:** ANOVA and Kruskal-Wallis both give p ≈ 0, and coal vs. gas (Welch t-test) p ≈ 10⁻¹⁰⁹: **mean capacity differs significantly between fuel types**.
5. **The fuel mix over time:** hydro was ~47% of new capacity in the 1950s and fell to ~9–12% by the 2000s–2010s. Coal was the largest share of new capacity from the 1970s to the 2010s (~40–50%), and nuclear peaked in the 1970s–80s (~11–19%). From the 2010s, **wind and solar grew fast**, and in the (small, ~150-plant) 2020 sample they make up ~75% of new capacity.
6. **Geography:** the map shows dense clusters in the USA, Europe, India and East Asia, with large coal plants concentrated in Asia, and many small solar/wind plants in Europe and the USA.
7. **Matrix analysis:** capacity and generation are almost perfectly correlated (r = 0.96, on log scale). The first eigenvector, a "plant size" factor (capacity + generation + being fossil-fuelled), explains ~43% of the variance; the second mainly contrasts latitude with commissioning year. One eigenvalue is close to 0 (0.04), showing that capacity and generation carry nearly the same information.
8. **New large renewables:** of the 52 renewable plants over 500 MW commissioned after 2010, about half are in **China**.

**Data caveats:** commissioning year is missing for half the plants, so the time analysis covers only the plants where it's known; the 2020 "decade" has very few plants; and generation figures for 2017 are estimates.